In [1]:
!pip install librosa -q
import librosa
import numpy as np
from pathlib import Path
import json
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

INPUT_DIR  = '/content/drive/MyDrive/lung_sound/smartphone_converted'
STETHO_DIR = '/content/drive/MyDrive/lung_sound/stethoscope'
OUTPUT_DIR = '/content/drive/MyDrive/lung_sound/processed/A'
LABEL_PATH = '/content/drive/MyDrive/lung_sound/processed/labels.json'

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('경로 설정 완료!')

Mounted at /content/drive
경로 설정 완료!


In [2]:
SR         = 22050
DURATION   = 5
N_MELS     = 128
HOP_LENGTH = 512

LABEL_MAP   = {(0,0): 0, (0,1): 1, (1,0): 1, (1,1): 1}
LABEL_NAMES = ['정상', '비정상']
print(f'샘플링 레이트: {SR}Hz, 구간: {DURATION}초')

샘플링 레이트: 22050Hz, 구간: 5초


In [3]:
def wav_to_mel(y, sr=SR):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, hop_length=HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
    return mel_db.astype(np.float32)

def get_label_for_segment(txt_path, seg_start, seg_end):
    label_duration = {0: 0, 1: 0}
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            start, end = float(parts[0]), float(parts[1])
            wheeze, crackle = int(parts[2]), int(parts[3])
            label = LABEL_MAP.get((wheeze, crackle), 0)
            overlap = max(0, min(end, seg_end) - max(start, seg_start))
            label_duration[label] += overlap
    return max(label_duration, key=label_duration.get)

print('함수 정의 완료!')

함수 정의 완료!


In [4]:
# ⚠️ 기존 폴더 비우기 (이전 v2~v4 잔재 섞이는 것 방지)
import shutil
if Path(OUTPUT_DIR).exists():
    shutil.rmtree(OUTPUT_DIR)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('출력 폴더 초기화 완료!')

출력 폴더 초기화 완료!


In [ ]:
wav_files = sorted(list(Path(INPUT_DIR).glob('*.wav')))
print(f'변환된 wav 파일 수: {len(wav_files)}')

all_labels = []
total = 0
segment_len = int(SR * DURATION)

for wav_path in wav_files:
    txt_path = Path(STETHO_DIR) / (wav_path.stem + '.txt')
    if not txt_path.exists():
        continue
    y, sr = librosa.load(str(wav_path), sr=SR)
    for idx, i in enumerate(range(0, len(y) - segment_len + 1, segment_len)):
        segment = y[i:i + segment_len]
        mel = wav_to_mel(segment)
        seg_start = i / SR
        seg_end = seg_start + DURATION
        label = get_label_for_segment(str(txt_path), seg_start, seg_end)
        save_name = f'{wav_path.stem}_{idx:04d}.npy'
        np.save(Path(OUTPUT_DIR) / save_name, mel)
        all_labels.append({'file': save_name, 'label': label})
        total += 1

with open(LABEL_PATH, 'w') as f:
    json.dump(all_labels, f)

print(f'\n✅ 전처리 완료! 총 {total}개')

변환된 wav 파일 수: 920

✅ 전처리 완료! 총 3901개
